# Initialization

Note: process_csv is where the majority of the code needs to be changed in order to create various datsets. Inside process_csv, stack_4ch_scalogram is called. Changing various inputs in this function call is how I generated the 4 different datasets (besides dataset1, that dataset is just less data). 

For dataset2, I used fmin = 4, fmax = 40, num_freq = 64, log_power = True, channel_first = True, normalize = "per_channel"
For dataset3, I used fmin = 4, fmax = 40, num_freq = 64, log_power = True, channel_first = True, normalize = "none"
For dataset2, I used fmin = 0.5, fmax = 40, num_freq = 64, log_power = True, channel_first = True, normalize = "none"

In [1]:
import os, re, glob
from pathlib import Path

import numpy as np
import pywt
from scipy.signal import detrend
from PIL import Image
import os
import matplotlib.cm as cm
import matplotlib
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ZIP_DIR="/content/drive/MyDrive/EEG_Data"
# ZIP_NAME="Cal-Poly-EEG-Data-main.zip"
# OUT_DIR="/content/drive/MyDrive/EEG_Data"

# !mkdir -p "$OUT_DIR"
# !unzip -q -o "$ZIP_DIR/$ZIP_NAME" -d "$OUT_DIR"
# !ls -lah "$OUT_DIR"


unzip:  cannot find or open /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main.zip, /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main.zip.zip or /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main.zip.ZIP.
total 4.0K
drwx------ 3 root root 4.0K Nov  6 18:33 Cal-Poly-EEG-Data-main


In [ ]:
# import os
# import zipfile

# base_dir = '/content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw'

# for root, dirs, files in os.walk(base_dir):
#     for file in files:
#         if file.endswith('.zip'):
#             zip_path = os.path.join(root, file)
#             extract_path = root # Extract to the current directory
#             with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#                 zip_ref.extractall(extract_path)
#             print(f"Unzipped {zip_path} to {extract_path}")

Unzipped /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2/growth/67ca1d0f5dd9e833402133a2.zip to /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2/growth
Unzipped /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2/growth/67c776a58cb8fc578866372c.zip to /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2/growth
Unzipped /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2/growth/67c6688bf0dce1020cccb08b.zip to /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2/growth
Unzipped /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2/growth/67c2836837717e8573ee0670.zip to /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2/growth
Unzipped /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2/growth/67ca1af61a49e79ec63fcc19.zip to /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2/growth
Unzipped /content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2/growth/680ac76059c825dbf7c315de.zip to /conte

# Helpers

In [3]:
# This function basically takes in the eeg signal, the specified window_size and stride and returns an
# array that has the various windows
def window_signal(sig, window_size, stride):
    starts = np.arange(0, max(len(sig) - window_size + 1, 0), stride)
    return np.stack([sig[s:s+window_size] for s in starts], axis=0) if len(starts) else np.empty((0, window_size))
    # Just a total windows by window size array of eeg values (n_windows, window_size)

"""def cwt_tile(signal_1d, fs, fmin=4, fmax=40, num_freqs=64, log_power=True):
    signal_1d = detrend(signal_1d, type="linear") # gets rid of linear drift (not sure if helpful especially since its per window)

    target_freqs = np.geomspace(fmin, fmax, num_freqs) # num_freqs from fmin to fmax. E.G. 64 freqs between 4 to 40 Hz
    central = pywt.central_frequency("morl")
    scales = central * fs / target_freqs # maps freq to CWT scales for Morlet

    coeffs, freqs = pywt.cwt(signal_1d, scales=scales, wavelet="morl", sampling_period=1/fs)
    power = (coeffs.real**2 + coeffs.imag**2) if np.iscomplexobj(coeffs) else np.abs(coeffs)**2

    # for when we want log_power for easier viewing
    if log_power:
        power = np.log10(power + 1e-12)

    # just filters outliers so the graphs are not dominated by outliers
    lo, hi = np.percentile(power, [5, 99])
    tile = np.clip((power - lo) / (hi - lo + 1e-12), 0, 1)

    return tile.astype(np.float32), freqs
    # returns scaolograms as tiles power x freq x time"""

# resizes images to 64, 64 -- mostly for if num_freq is changed to smth other than 64, not sure if resizing is the move here tho, will test later
def resize_tile(tile, out_size=(64, 64)):
    # PIL expects HxW, so treat F as H and T as W
    arr = (tile * 255.0).astype(np.uint8)
    #im = Image.fromarray(arr, mode="L")
    im = Image.fromarray(arr.astype(np.uint8))
    im = im.resize(out_size, Image.BILINEAR)
    return np.asarray(im, dtype=np.uint8)

"""
# for visualization
# saves a colored png
def save_tile_png(tile_float, out_path, cmap_name = "viridis"):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    #cmap = cm.get_cmap(cmap_name)
    cmap = matplotlib.colormaps.get_cmap(cmap_name)
    colored = (cmap(tile_float)[..., :3] * 255).astype(np.uint8)
    #Image.fromarray(colored, mode="RGB").save(out_path, optimize=True)
    Image.fromarray(colored).convert("RGB").save(out_path, optimize=True)
"""

# Turning the 1D signal into a 2D scalogram
def cwt_full_signal(signal_1d, fs, fmin=0.5, fmax=40, num_freqs=64, log_power=True, mode="none"):
    signal_1d = detrend(signal_1d, type="linear") # gets rid of linear drift (not sure if helpful especially since its per window)

    target_freqs = np.geomspace(fmin, fmax, num_freqs) # num_freqs from fmin to fmax. E.G. 64 freqs between 4 to 40 Hz
    central = pywt.central_frequency("morl")
    scales = central * fs / target_freqs # maps freq to CWT scales for Morlet

    coeffs, freqs = pywt.cwt(signal_1d, scales=scales, wavelet="morl", sampling_period=1/fs) # actual continous wavelet transform. Coeffs: wavelet coefficients, freqs associated with those rows
    power = (coeffs.real**2 + coeffs.imag**2) if np.iscomplexobj(coeffs) else np.abs(coeffs)**2 # convert coefficients to power

    # for when we want log_power for easier viewing
    if log_power:
        power = np.log10(power + 1e-12)


    if mode == "per_channel":
        # just filters outliers so the graphs are not dominated by outliers
        lo, hi = np.percentile(power, [5, 99])
        tile = np.clip((power - lo) / (hi - lo + 1e-12), 0, 1)
        return tile.astype(np.float32), freqs

    elif mode != "none":
        raise ValueError(f"Not defined: {mode}")

    # returns scaolograms as tiles power x freq x time
    return power.astype(np.float32), freqs


# Take in 4 signals and turn into one stacked tensor
# ch_signals: list of 4 channel signals
def stack_4ch_scalogram(ch_signals, fs, fmin=0.5, fmax=40, num_freqs=64, log_power=True,
                        channels_first=True, normalize="none"):
    # init container
    tiles = []
    freqs_out = None
    min_T = None

    # looping through each channel
    for sig in ch_signals:
        # compute scalogram for each channel
        tile, freqs = cwt_full_signal(
            sig, fs,
            fmin=fmin, fmax=fmax, num_freqs=num_freqs,
            log_power=log_power,
            mode=normalize
        )
        if freqs_out is None:
            freqs_out = freqs
        min_T = tile.shape[1] if min_T is None else min(min_T, tile.shape[1]) # tracking shortest time dimension
        tiles.append(tile)

    # align time dims (crop to shortest)
    tiles = [t[:, :min_T] for t in tiles]  # each (F, T)

    X = np.stack(tiles, axis=0).astype(np.float32)  # (4, F, T) ---- basically just stacks

    # global normalization across all channels
    if normalize == "global":
        lo, hi = np.percentile(X, [5, 99])
        X = np.clip((X - lo) / (hi - lo + 1e-12), 0, 1).astype(np.float32)

    # can optionally move channels to the end depending on what output is needed
    if not channels_first:
        X = np.transpose(X, (1, 2, 0))  # (F, T, 4)

    return X, freqs_out

"""
# Takes dataframe with EEG time series and cuts each channel into windows then computes a stacked scalogram for each window.
def windowed_stacked_scalograms(df, channel_names, fs, window_size, stride, fmin=4,
                                fmax=40, num_freqs=64, log_power=True, normalize="per_channel"):
    # window each channel
    wins_by_ch = []
    n_wins = None
    for ch in channel_names:
        sig = np.asarray(df[ch], dtype=float).reshape(-1)
        wins = window_signal(sig, window_size, stride)  # (n_windows, window_size)
        if n_wins is None:
            n_wins = wins.shape[0]
        else:
            n_wins = min(n_wins, wins.shape[0])
        wins_by_ch.append(wins)

    # truncate all channels to same number of windows
    wins_by_ch = [w[:n_wins] for w in wins_by_ch]

    # compute stacked scalogram per window
    out = []
    for i in range(n_wins):
        ch_signals = [wins_by_ch[c][i] for c in range(4)]
        X, freqs = stack_4ch_scalogram(
            ch_signals, fs, fmin=fmin, fmax=fmax, num_freqs=num_freqs,
            log_power=log_power, channels_first=True, normalize=normalize
        )
        out.append(X)

    return np.stack(out, axis=0), freqs  # (n_windows, 4, F, T)

"""






'\n# Takes dataframe with EEG time series and cuts each channel into windows then computes a stacked scalogram for each window.\ndef windowed_stacked_scalograms(df, channel_names, fs, window_size, stride, fmin=4,\n                                fmax=40, num_freqs=64, log_power=True, normalize="per_channel"):\n    # window each channel\n    wins_by_ch = []\n    n_wins = None\n    for ch in channel_names:\n        sig = np.asarray(df[ch], dtype=float).reshape(-1)\n        wins = window_signal(sig, window_size, stride)  # (n_windows, window_size)\n        if n_wins is None:\n            n_wins = wins.shape[0]\n        else:\n            n_wins = min(n_wins, wins.shape[0])\n        wins_by_ch.append(wins)\n\n    # truncate all channels to same number of windows\n    wins_by_ch = [w[:n_wins] for w in wins_by_ch]\n\n    # compute stacked scalogram per window\n    out = []\n    for i in range(n_wins):\n        ch_signals = [wins_by_ch[c][i] for c in range(4)]\n        X, freqs = stack_4ch_sc

# Load in the Data

In [4]:
# --- Configuration ---
data_roots = [
    '/content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/1',
    '/content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2'
    ]         # Change to 'OrganizedRSCA/2' for study 2
output_root = 'MergedRSCA'            # Output directory for merged data
MERGE_ALL = False                      # <<< Toggle: True for full merge only, False for group-by-group

# --- Define question groups by index range ---
all_qs = range(1,33)

question_groups = {
  f"Question{q:02d}": [q] for q in all_qs
}

for root in data_roots:
    root_label = os.path.basename(root)
    # out_root_for_study = os.path.join(output_root, root_label)

    if not os.path.isdir(root):
        print(f"Warning root folder not found: {root}")
        continue

    print(f"\n======== Processing study {root_label}==========")

    for study_folder in os.listdir(root):
        study_path = os.path.join(root, study_folder)

        if not os.path.isdir(study_path):
            continue

        output_study_path = os.path.join(output_root, root_label, study_folder)
        print(f"\n =======Study: {study_folder}=======")

        for person_folder in os.listdir(study_path):
            person_path = os.path.join(study_path, person_folder)

            if not os.path.isdir(person_path):
                continue

            csv_files = sorted(glob.glob(os.path.join(person_path, '*.csv')))
            file_map = {}

            for file in csv_files:
                parts = os.path.basename(file).split('_')
                if len(parts) >= 4:
                    try:
                        seq_num = int(parts[2])  # Assumes format like: PREFIX_<ID>_<QNUM>_SUFFIX.csv
                        file_map[seq_num] = file
                    except ValueError:
                        continue

            person_id = os.path.basename(person_path)
            merged_out_dir = os.path.join(output_study_path, person_id)
            os.makedirs(merged_out_dir, exist_ok=True)

            if MERGE_ALL:
                # --- Merge all files into one, skip group merging ---
                all_dataframes = []
                # Sort file_map by key (sequence number) before processing
                for seq_num in sorted(file_map.keys()):
                    file = file_map[seq_num]
                    df = pd.read_csv(file)
                    all_dataframes.append(df)

                if all_dataframes:
                    full_merged = pd.concat(all_dataframes, ignore_index=True)
                    full_merged.dropna(inplace=True)

                    full_merged_path = os.path.join(merged_out_dir, f"{person_id}_all.csv")
                    full_merged.to_csv(full_merged_path, index=False)
                    print(f"Saved merged all file: {full_merged_path}")
            else:
                # --- Group-by-group merge ---
                all_missing = {}

                for group_name, questions in question_groups.items():
                    missing = []
                    dataframes = []

                    # Sort question numbers before processing
                    for q_num in sorted(questions):
                        if q_num not in file_map:
                            missing.append(q_num)
                        else:
                            df = pd.read_csv(file_map[q_num])
                            dataframes.append(df)

                    if missing:
                        all_missing[group_name] = missing
                        print(f"Skipped {group_name} for {person_id} due to missing files: {missing}")
                        continue

                    if dataframes:
                        merged_df = pd.concat(dataframes, ignore_index=True)
                        merged_df.dropna(inplace=True)

                        merged_path = os.path.join(merged_out_dir, f"{person_id}_{group_name}.csv")
                        merged_df.to_csv(merged_path, index=False)
                        print(f"Saved merged file: {merged_path}")

                # --- Save missing summary ---
                if all_missing:
                    missing_path = os.path.join(merged_out_dir, f"{person_id}_missing_questions.txt")
                    with open(missing_path, 'w') as f:
                        for group, missing_list in all_missing.items():
                            f.write(f"{group}: missing question numbers {missing_list}\n")


======== Processing study 1==========

 =======Study: control=======
Skipped Question01 for 67c26f8829d462ae823bb2c2 due to missing files: [1]
Skipped Question02 for 67c26f8829d462ae823bb2c2 due to missing files: [2]
Skipped Question03 for 67c26f8829d462ae823bb2c2 due to missing files: [3]
Skipped Question04 for 67c26f8829d462ae823bb2c2 due to missing files: [4]
Skipped Question05 for 67c26f8829d462ae823bb2c2 due to missing files: [5]
Skipped Question06 for 67c26f8829d462ae823bb2c2 due to missing files: [6]
Skipped Question07 for 67c26f8829d462ae823bb2c2 due to missing files: [7]
Skipped Question08 for 67c26f8829d462ae823bb2c2 due to missing files: [8]
Skipped Question09 for 67c26f8829d462ae823bb2c2 due to missing files: [9]
Skipped Question10 for 67c26f8829d462ae823bb2c2 due to missing files: [10]
Skipped Question11 for 67c26f8829d462ae823bb2c2 due to missing files: [11]
Skipped Question12 for 67c26f8829d462ae823bb2c2 due to missing files: [12]
Skipped Question13 for 67c26f8829d462ae

# Generating Dataset


In [14]:
DATA_ROOT = Path("/content/MergedRSCA")
STUDY = ["1", "2"]

OUT_ROOT = Path("/content/drive/MyDrive/Scalograms_numpy_per_question_stacked_noNorm4")
OUT_ROOT.mkdir(parents=True, exist_ok=True)


GROUPS = ["control", "fixed", "growth"]

# ON CHANNEL N
fs = 256
WINDOW_SIZE = 64 #idk why 64
STRIDE = 32 # 50% overlap so half of whats in cur is old

channel_names = ["Channel 1", "Channel 2", "Channel 3", "Channel 4"]

In [15]:
csv_name_format = re.compile(r".*_Question(\d+)\.csv$", re.IGNORECASE)


### More Helpers

In [13]:
# ----------------------------
# HELPERS, these ones for dealing with specific structure of how data loaded in
# ----------------------------

# gives the immediate subfolders. Used for yielding participant directories (immediate children that are directories) -> look inside a group folder and give me each participant folder
def iter_participants(group_dir: Path):
    for p in sorted(group_dir.iterdir()):
        if p.is_dir():
            yield p

# Finds all CSV files udner a particpant folder
def list_question_csvs(person_dir: Path):
    # Since CSVs are inside the participant folder; recursive handles subfolders too.
    return sorted(person_dir.rglob("*.csv")) # Return sorted list of CSV paths under participant directory (recursive)

"""# Builds output folder for saving per channel results
def output_dir_for(group: str, participant_id: str, q_num: int, channel: str) -> Path:
    channel_folder = channel.replace(" ", "_") # formatting e.g channel 1 -> channel_1
    return OUT_ROOT / group / participant_id / f"Question{q_num:02d}" / channel_folder # builds a path"""

# Save each tile as winXXXXX.npy. Incorperates auto-resume: skip any file that already exists.
def save_tiles_resume(tiles, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True) # making sure parent dir exist

    # loop through each tile
    for i, tile in enumerate(tiles):
        out_path = out_dir / f"win{i:05d}.npy"
        if out_path.exists():
            continue  # this adds in resume behavior

        tile_f32 = tile.astype(np.float32) / 255.0
        np.save(out_path, tile_f32)

"""# Saves stacked multi channel scalograms for each window in its own .npy file
# e.g X_windows: (n_windows, 4, F, T) float32 in [0,1] (if normalized) -> Saves: winXXXXX.npy each containing (4, F, T)
def save_windows_stacked_resume(X_windows, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True) # making sure parent dir exists

    # loop over windows
    for i in range(X_windows.shape[0]):
        out_path = out_dir / f"win{i:05d}.npy"
        if out_path.exists():
            continue # resume behavior
        np.save(out_path, X_windows[i].astype(np.float32))"""

# read 1 csv for 1 question and compute the stacked 4 channel scalogram and save as "stacked_full.npy"
def process_csv(csv_path: Path, group: str, participant_id: str, mode="stacked_full"):
    base = csv_path.name # Get filename
    mat = csv_name_format.match(base) # certain pattern for csv name
    if not mat:
        return 0

    q_num = int(mat.group(1)) # extract question number
    df = pd.read_csv(csv_path) # create dataframe

    # checks mode (currently only stacked_full works was gonna incorperate the functions above)
    if mode != "stacked_full":
        raise ValueError("Use mode='stacked_full' for 1 file per question with 4 stacked channels.")

    chs = channel_names[:4] # get the 4 channels wanted

    # collect full signals for each channel
    ch_signals = []
    for ch in chs:
        if ch not in df.columns:
            print(f"[WARN] Missing '{ch}' in {csv_path} -> skipping this question")
            return 0
        ch_signals.append(np.asarray(df[ch], dtype=float).reshape(-1)) # convert into a 1D numpy array

    # compute stacked full scalogram: (4, F, T)
    X, freqs = stack_4ch_scalogram(
        ch_signals, fs,
        fmin=4, fmax=40, num_freqs=64,
        log_power=True,
        channels_first=True,
        normalize="none"   # or "global", "per_channel"
    )

    # save one file per question
    out_dir = OUT_ROOT / group / participant_id / f"Question{q_num:02d}"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / "stacked_full.npy"
    if out_path.exists():
        return 0

    np.save(out_path, X.astype(np.float32)) # save stacked tensor
    return 1

### Actual Generation

In [16]:
# tracking totals
total_saved = 0
total_csvs = 0

for study in STUDY:
    study_dir = DATA_ROOT / study
    if not study_dir.exists():
        print(f"[WARN] Study folder missing: {study_dir} (skipping)")
        continue

     # pritn what study is being evaluated
    print(f"\n=== STUDY: {study} ===")

    # lookign at each group (control, fixed, growth)
    for group in GROUPS:
        group_dir = study_dir / group # build a path to given group

        # if the group does not exist, skip it (saftey check)
        if not group_dir.exists():
            print(f"[WARN] Group folder missing: {group_dir} (skipping)")
            continue

        # pritn what group is being evaluated
        print(f"\n=== GROUP: {group} ===")

        # loop throguh participant folders in the group folder
        for person_dir in iter_participants(group_dir):
            participant_id = person_dir.name # extract participant ID
            csv_paths = list_question_csvs(person_dir) # get all csvs for that participant

            # print info
            print(f"\nParticipant {participant_id}: found {len(csv_paths)} CSV files")

            # loop through each csv
            for csv_path in csv_paths:
                # Only Question CSVs will be processed; others ignored
                saved_now = process_csv(csv_path, group, participant_id) # converts csvs into scalgorams (.npy format)
                if saved_now is not None:
                    total_saved += saved_now
                total_csvs += 1

print("\nDONE")
print(f"Total CSV files scanned: {total_csvs}")
print(f"Total new images saved (approx): {total_saved}")
print(f"Output root: {OUT_ROOT}")


=== STUDY: 1 ===

=== GROUP: control ===

Participant 67c26f8829d462ae823bb2c2: found 0 CSV files

Participant 67c6544e496c87641bc32d1e: found 32 CSV files

Participant 67c6621d2d2c5367115b87c6: found 32 CSV files

Participant 67c66f62f0dce1020cccb8ba: found 32 CSV files

Participant 67c77b63037bc8de9f8902ca: found 32 CSV files

Participant 67c77fa5037bc8de9f890800: found 32 CSV files

Participant 67c786dcbe53b42f18bcc366: found 32 CSV files

Participant 67ca2115c434fe4ee537e752: found 32 CSV files

Participant 67d09d8c1594c6eaac88c03f: found 0 CSV files

Participant 67d0a4ca982703777809a321: found 0 CSV files

Participant 67d0a6e383e9a21f7033690d: found 0 CSV files

Participant 68082161332567ffa5214e0b: found 32 CSV files

Participant 6808335d332567ffa52165f8: found 32 CSV files

Participant 680acc8a59c825dbf7c31deb: found 32 CSV files

Participant 681144c8bb134e569280d496: found 32 CSV files

=== GROUP: fixed ===

Participant 67c271ffe81e2cd37b70360b: found 32 CSV files

Participant